# Stock Return Regression

**Load and inspect the data**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

feature_data = pd.read_csv(
    PROCESSED_DIR / "lseg_feature_engineered.csv",
    parse_dates=["Date"]
)

price_data = pd.read_csv(
    PROCESSED_DIR / "lseg_ohlcv_clean.csv",
    parse_dates=["Date"]
)

feature_columns = [
    "returns", "returns_lag_1", "returns_lag_2",
    "momentum_5", "momentum_10",
    "daily_range", "candle_body",
    "sma_5_ratio", "sma_10_ratio", "sma_20_ratio",
    "volatility_5", "volatility_10", "volatility_20",
    "rsi", "macd", "macd_signal", "macd_diff",
    "volume_change", "volume_ratio_20",
    "bb_high", "bb_low", "bb_width", "skewness_20"
]

required_columns = [
    "Ticker", "Country", "Date", "Segment_ID"
]

assert all(
    column in feature_data.columns
    for column in required_columns + feature_columns
)

assert all(
    column in price_data.columns
    for column in required_columns
)

assert not feature_data.duplicated(["Ticker", "Date"]).any()
assert not price_data.duplicated(["Ticker", "Date"]).any()

print("=== DATASET SUMMARY ===")
print("Feature observations:", len(feature_data))
print("Price observations:", len(price_data))
print("Stocks:", feature_data["Ticker"].nunique())
print("Technical features:", len(feature_columns))
print("Earliest date:", feature_data["Date"].min().date())
print("Latest date:", feature_data["Date"].max().date())

print("\n=== OBSERVATIONS BY MARKET ===")
display(
    feature_data.groupby("Country").agg(
        Stocks=("Ticker", "nunique"),
        Observations=("Date", "size")
    )
)

print("\n=== AVAILABLE PRICE COLUMNS ===")
print(
    [
        column for column in price_data.columns
        if "close" in column.lower()
    ]
)

print("\n=== PRICE DATA COLUMNS ===")
print(price_data.columns.tolist())

print("\nCompleted.")

=== DATASET SUMMARY ===
Feature observations: 37798
Price observations: 37798
Stocks: 13
Technical features: 23
Earliest date: 2015-01-02
Latest date: 2026-10-30

=== OBSERVATIONS BY MARKET ===


,Stocks,Observations
Country,,
Malaysia,10,28894
Singapore,3,8904



=== AVAILABLE PRICE COLUMNS ===
['Close']

=== PRICE DATA COLUMNS ===
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Name', 'Country', 'LSEG_RIC', 'Segment_ID', 'Before_Gap', 'After_Gap', 'returns']

Completed.


**Generate one-day and five-day forward return targets**

In [2]:
DEVELOPMENT_END = pd.Timestamp("2024-12-31")

# Restrict target construction to the development period
development_features = feature_data.loc[
    feature_data["Date"] <= DEVELOPMENT_END
].copy()

development_prices = price_data.loc[
    price_data["Date"] <= DEVELOPMENT_END,
    ["Ticker", "Segment_ID", "Date", "Close"]
].copy()

development_prices = development_prices.rename(
    columns={"Close": "Entry_Close"}
)

# Match historical features to closing prices
regression_data = development_features.merge(
    development_prices,
    on=["Ticker", "Segment_ID", "Date"],
    how="left",
    validate="one_to_one"
)

assert len(regression_data) == len(development_features)
assert regression_data["Entry_Close"].notna().all()
assert (regression_data["Entry_Close"] > 0).all()

regression_data = regression_data.sort_values(
    ["Ticker", "Segment_ID", "Date"]
).reset_index(drop=True)

# Create future returns within uninterrupted price segments
groups = regression_data.groupby(
    ["Ticker", "Segment_ID"],
    sort=False
)

for horizon in [1, 5]:

    future_close = groups["Entry_Close"].shift(-horizon)
    future_date = groups["Date"].shift(-horizon)

    return_column = f"Target_Return_{horizon}D"
    date_column = f"Target_End_Date_{horizon}D"

    regression_data[return_column] = (
        future_close / regression_data["Entry_Close"] - 1
    )

    regression_data[date_column] = future_date

    valid_target = regression_data[return_column].notna()

    assert (
        regression_data.loc[valid_target, date_column]
        > regression_data.loc[valid_target, "Date"]
    ).all()

    assert (
        regression_data[return_column].notna()
        == regression_data[date_column].notna()
    ).all()

# Identify observations with complete technical features
regression_data["Feature_Complete"] = (
    regression_data[feature_columns].notna().all(axis=1)
)

# Record target availability without removing observations
regression_data["Target_1D_Available"] = (
    regression_data["Target_Return_1D"].notna()
)

regression_data["Target_5D_Available"] = (
    regression_data["Target_Return_5D"].notna()
)

# Summarise target availability
target_summary = (
    regression_data.groupby("Country")
    .agg(
        Stocks=("Ticker", "nunique"),
        Observations=("Date", "size"),
        Complete_Features=("Feature_Complete", "sum"),
        Available_1D=("Target_1D_Available", "sum"),
        Available_5D=("Target_5D_Available", "sum")
    )
    .reset_index()
)

print("=== REGRESSION TARGET SUMMARY ===")
display(target_summary)

print("\n=== FORWARD RETURN EXAMPLES ===")

display(
    regression_data[
        [
            "Ticker",
            "Date",
            "Entry_Close",
            "Target_End_Date_1D",
            "Target_Return_1D",
            "Target_End_Date_5D",
            "Target_Return_5D"
        ]
    ].dropna().head(10).round(6)
)

print("\n=== DEVELOPMENT DATE RANGE ===")
print("Earliest date:", regression_data["Date"].min().date())
print("Latest date:", regression_data["Date"].max().date())
print("Observations:", len(regression_data))

# Save development targets for subsequent regression modelling
regression_data.to_csv(
    PROCESSED_DIR / "lseg_regression_targets_development.csv",
    index=False
)

print("\nCompleted.")

=== REGRESSION TARGET SUMMARY ===


,Country,Stocks,Observations,Complete_Features,Available_1D,Available_5D
0,Malaysia,10,24424,23518,24396,24284
1,Singapore,3,7521,7422,7518,7506



=== FORWARD RETURN EXAMPLES ===


C:\Users\zechi\AppData\Local\Temp\ipykernel_10732\1133411936.py:108: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].dropna().head(10).round(6)


,Ticker,Date,Entry_Close,Target_End_Date_1D,Target_Return_1D,Target_End_Date_5D,Target_Return_5D
0,1015.KL,2015-01-02,6.54,2015-01-05,-0.015291,2015-01-09,-0.018349
1,1015.KL,2015-01-05,6.44,2015-01-06,-0.021739,2015-01-12,0.000000
2,1015.KL,2015-01-06,6.30,2015-01-07,0.004762,2015-01-13,0.023810
3,1015.KL,2015-01-07,6.33,2015-01-08,0.015798,2015-01-14,0.012638
4,1015.KL,2015-01-08,6.43,2015-01-09,-0.001555,2015-01-15,-0.004666
5,1015.KL,2015-01-09,6.42,2015-01-12,0.003115,2015-01-16,-0.001558
6,1015.KL,2015-01-12,6.44,2015-01-13,0.001553,2015-01-19,0.000000
7,1015.KL,2015-01-13,6.45,2015-01-14,-0.006202,2015-01-20,0.004651
8,1015.KL,2015-01-14,6.41,2015-01-15,-0.001560,2015-01-21,0.015601
9,1015.KL,2015-01-15,6.40,2015-01-16,0.001562,2015-01-22,0.029687



=== DEVELOPMENT DATE RANGE ===
Earliest date: 2015-01-02
Latest date: 2024-12-31
Observations: 31945

Completed.


**Prepare regression training and validation splits**

In [3]:
TRAIN_END = pd.Timestamp("2022-12-31")
VALIDATION_START = pd.Timestamp("2023-01-01")
VALIDATION_END = pd.Timestamp("2024-12-31")

regression_split_records = []

for horizon in [1, 5]:

    target_column = f"Target_Return_{horizon}D"
    end_column = f"Target_End_Date_{horizon}D"

    # Retain complete features and observable targets
    eligible = regression_data.loc[
        regression_data["Feature_Complete"]
        & regression_data[target_column].notna()
        & regression_data[end_column].notna()
    ].copy()

    eligible = eligible.loc[
        np.isfinite(eligible[target_column])
    ].copy()

    eligible["Horizon_Days"] = horizon
    eligible["Target_Return"] = eligible[target_column]
    eligible["Target_End_Date"] = eligible[end_column]

    # Separate observations using complete target horizons
    train_mask = (
        (eligible["Date"] <= TRAIN_END)
        & (eligible["Target_End_Date"] <= TRAIN_END)
    )

    validation_mask = (
        (eligible["Date"] >= VALIDATION_START)
        & (eligible["Date"] <= VALIDATION_END)
        & (eligible["Target_End_Date"] <= VALIDATION_END)
    )

    selected = eligible.loc[
        train_mask | validation_mask
    ].copy()

    selected["Split"] = np.where(
        train_mask.loc[selected.index],
        "Train_2015_2022",
        "Validation_2023_2024"
    )

    regression_split_records.append(
        selected[
            [
                "Ticker",
                "Country",
                "Date",
                "Segment_ID",
                "Entry_Close",
                "Horizon_Days",
                "Target_End_Date",
                "Target_Return",
                "Split"
            ] + feature_columns
        ]
    )

# Combine the two regression horizons
regression_model_data = pd.concat(
    regression_split_records,
    ignore_index=True
)

regression_model_data = regression_model_data.sort_values(
    ["Horizon_Days", "Country", "Ticker", "Date"]
).reset_index(drop=True)

# Validate split integrity
assert not regression_model_data.duplicated(
    ["Ticker", "Date", "Horizon_Days"]
).any()

assert regression_model_data[feature_columns].notna().all().all()

assert (
    regression_model_data["Target_End_Date"]
    > regression_model_data["Date"]
).all()

train_data = regression_model_data.loc[
    regression_model_data["Split"] == "Train_2015_2022"
]

validation_data = regression_model_data.loc[
    regression_model_data["Split"] == "Validation_2023_2024"
]

assert (train_data["Target_End_Date"] <= TRAIN_END).all()

assert (
    validation_data["Date"] >= VALIDATION_START
).all()

assert (
    validation_data["Target_End_Date"] <= VALIDATION_END
).all()

# Summarise the number of eligible observations
split_summary = (
    regression_model_data.groupby(
        ["Country", "Horizon_Days", "Split"]
    )
    .agg(
        Stocks=("Ticker", "nunique"),
        Observations=("Date", "size"),
        First_Date=("Date", "min"),
        Last_Date=("Date", "max")
    )
    .reset_index()
)

print("=== REGRESSION TRAINING AND VALIDATION SPLITS ===")
display(split_summary)

# Examine target distributions
distribution_records = []

for (country, horizon, split), group in (
    regression_model_data.groupby(
        ["Country", "Horizon_Days", "Split"]
    )
):

    returns = group["Target_Return"]

    distribution_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "Split": split,
        "Observations": len(returns),
        "Mean_Return": returns.mean(),
        "Median_Return": returns.median(),
        "Std_Return": returns.std(),
        "Zero_Return_Share": np.mean(returns == 0),
        "Mean_Absolute_Return": returns.abs().mean(),
        "Absolute_Return_P99": returns.abs().quantile(0.99),
        "Maximum_Absolute_Return": returns.abs().max()
    })

return_distribution = pd.DataFrame(
    distribution_records
)

print("\n=== REGRESSION TARGET DISTRIBUTIONS ===")
display(return_distribution.round(6))

# Identify extreme forward returns
extreme_returns = (
    regression_model_data.assign(
        Absolute_Return=lambda df: df["Target_Return"].abs()
    )
    .sort_values("Absolute_Return", ascending=False)
    .head(15)
)

print("\n=== LARGEST ABSOLUTE FORWARD RETURNS ===")

display(
    extreme_returns[
        [
            "Country",
            "Ticker",
            "Date",
            "Horizon_Days",
            "Target_End_Date",
            "Target_Return",
            "Split"
        ]
    ].round({"Target_Return": 6})
)

# Save regression development dataset
regression_model_data.to_csv(
    PROCESSED_DIR / "lseg_regression_model_development.csv",
    index=False
)

print("\nCompleted.")

=== REGRESSION TRAINING AND VALIDATION SPLITS ===


,Country,Horizon_Days,Split,Stocks,Observations,First_Date,Last_Date
0,Malaysia,1,Train_2015_2022,10,18618,2015-02-24,2022-12-29
1,Malaysia,1,Validation_2023_2024,10,4864,2023-01-03,2024-12-30
2,Malaysia,5,Train_2015_2022,10,18518,2015-02-24,2022-12-22
3,Malaysia,5,Validation_2023_2024,10,4820,2023-01-03,2024-12-23
4,Singapore,1,Train_2015_2022,3,5916,2015-02-18,2022-12-29
5,Singapore,1,Validation_2023_2024,3,1500,2023-01-03,2024-12-30
6,Singapore,5,Train_2015_2022,3,5904,2015-02-18,2022-12-22
7,Singapore,5,Validation_2023_2024,3,1488,2023-01-03,2024-12-23



=== REGRESSION TARGET DISTRIBUTIONS ===


,Country,Horizon_Days,Split,Observations,Mean_Return,Median_Return,Std_Return,Zero_Return_Share,Mean_Absolute_Return,Absolute_Return_P99,Maximum_Absolute_Return
0,Malaysia,1,Train_2015_2022,18618,0.000019,0.000000,0.014221,0.154420,0.009059,0.050350,0.231884
1,Malaysia,1,Validation_2023_2024,4864,0.000445,0.000000,0.011202,0.175987,0.007204,0.040844,0.126866
2,Malaysia,5,Train_2015_2022,18518,0.000083,0.000000,0.031415,0.059726,0.020626,0.113430,0.318841
3,Malaysia,5,Validation_2023_2024,4820,0.002136,0.000972,0.024271,0.071162,0.016379,0.091644,0.189873
4,Singapore,1,Train_2015_2022,5916,0.000244,0.000000,0.011737,0.030257,0.008395,0.038374,0.080435
5,Singapore,1,Validation_2023_2024,1500,0.000608,0.000000,0.009556,0.032667,0.006859,0.030559,0.071772
6,Singapore,5,Train_2015_2022,5904,0.001282,0.001163,0.028343,0.012703,0.020511,0.093738,0.174466
7,Singapore,5,Validation_2023_2024,1488,0.002942,0.001718,0.022527,0.010081,0.016535,0.077725,0.144240



=== LARGEST ABSOLUTE FORWARD RETURNS ===


,Country,Ticker,Date,Horizon_Days,Target_End_Date,Target_Return,Split
41468,Malaysia,1171.KL,2020-03-12,5,2020-03-19,-0.318841,Train_2015_2022
40596,Malaysia,1171.KL,2016-06-17,5,2016-06-27,-0.317112,Train_2015_2022
40624,Malaysia,1171.KL,2016-08-01,5,2016-08-08,0.314286,Train_2015_2022
40597,Malaysia,1171.KL,2016-06-20,5,2016-06-28,-0.309179,Train_2015_2022
41467,Malaysia,1171.KL,2020-03-11,5,2020-03-18,-0.287671,Train_2015_2022
40594,Malaysia,1171.KL,2016-06-15,5,2016-06-23,-0.281818,Train_2015_2022
40593,Malaysia,1171.KL,2016-06-14,5,2016-06-21,-0.279050,Train_2015_2022
40595,Malaysia,1171.KL,2016-06-16,5,2016-06-24,-0.276329,Train_2015_2022
34424,Malaysia,1023.KL,2020-03-12,5,2020-03-19,-0.264637,Train_2015_2022
34423,Malaysia,1023.KL,2020-03-11,5,2020-03-18,-0.236902,Train_2015_2022



Completed.


**Train initial XGBoost regression models**

In [4]:
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

regression_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

regression_models = {}
regression_result_records = []
regression_prediction_records = []

for country in ["Malaysia", "Singapore"]:

    for horizon in [1, 5]:

        # Select the training and validation observations
        train = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Train_2015_2022")
        ].copy()

        valid = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Validation_2023_2024")
        ].copy()

        X_train = train[feature_columns]
        X_valid = valid[feature_columns]

        # Express returns in percentage points
        y_train = train["Target_Return"].to_numpy() * 100
        y_valid = valid["Target_Return"].to_numpy() * 100

        assert len(X_train) > 0
        assert len(X_valid) > 0
        assert np.isfinite(X_train.to_numpy()).all()
        assert np.isfinite(X_valid.to_numpy()).all()
        assert np.isfinite(y_train).all()
        assert np.isfinite(y_valid).all()

        # Train XGBoost regression
        model = XGBRegressor(**regression_params)

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        xgb_predictions = model.predict(X_valid)

        regression_models[(country, horizon)] = model

        # Create constant forecasting baselines
        zero_predictions = np.zeros(len(valid))

        mean_predictions = np.full(
            len(valid),
            y_train.mean()
        )

        prediction_methods = {
            "XGBoost": xgb_predictions,
            "Zero_Return": zero_predictions,
            "Training_Mean": mean_predictions
        }

        # Evaluate all forecasting methods
        for method, predictions in prediction_methods.items():

            assert np.isfinite(predictions).all()

            regression_result_records.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Method": method,
                "Training_Rows": len(train),
                "Validation_Rows": len(valid),
                "MAE_pp": mean_absolute_error(
                    y_valid,
                    predictions
                ),
                "RMSE_pp": np.sqrt(
                    mean_squared_error(
                        y_valid,
                        predictions
                    )
                ),
                "R2": r2_score(
                    y_valid,
                    predictions
                )
            })

        # Save predictions as decimal returns
        prediction_output = valid[
            [
                "Ticker",
                "Country",
                "Date",
                "Horizon_Days",
                "Target_End_Date",
                "Target_Return"
            ]
        ].copy()

        prediction_output["XGBoost_Predicted_Return"] = (
            xgb_predictions / 100
        )

        prediction_output["Zero_Predicted_Return"] = (
            zero_predictions / 100
        )

        prediction_output["Mean_Predicted_Return"] = (
            mean_predictions / 100
        )

        regression_prediction_records.append(
            prediction_output
        )

        print(
            f"{country} | {horizon}-day return | "
            f"Training: {len(train)} | "
            f"Validation: {len(valid)}"
        )

# Combine results
regression_results_df = pd.DataFrame(
    regression_result_records
)

regression_validation_predictions = pd.concat(
    regression_prediction_records,
    ignore_index=True
)

regression_results_df = (
    regression_results_df
    .sort_values(
        ["Country", "Horizon_Days", "MAE_pp"]
    )
    .reset_index(drop=True)
)

print("\n=== INITIAL REGRESSION RESULTS ===")

display(
    regression_results_df[
        [
            "Country",
            "Horizon_Days",
            "Method",
            "Validation_Rows",
            "MAE_pp",
            "RMSE_pp",
            "R2"
        ]
    ].round(5)
)

# Compare XGBoost against the zero-return baseline
comparison_records = []

for (country, horizon), group in regression_results_df.groupby(
    ["Country", "Horizon_Days"]
):

    xgb = group.loc[
        group["Method"] == "XGBoost"
    ].iloc[0]

    zero = group.loc[
        group["Method"] == "Zero_Return"
    ].iloc[0]

    mean = group.loc[
        group["Method"] == "Training_Mean"
    ].iloc[0]

    comparison_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "XGBoost_MAE_pp": xgb["MAE_pp"],
        "Zero_MAE_pp": zero["MAE_pp"],
        "Mean_MAE_pp": mean["MAE_pp"],
        "MAE_Improvement_vs_Zero_pct": (
            (zero["MAE_pp"] - xgb["MAE_pp"])
            / zero["MAE_pp"] * 100
        ),
        "XGBoost_RMSE_pp": xgb["RMSE_pp"],
        "Zero_RMSE_pp": zero["RMSE_pp"],
        "RMSE_Improvement_vs_Zero_pct": (
            (zero["RMSE_pp"] - xgb["RMSE_pp"])
            / zero["RMSE_pp"] * 100
        ),
        "XGBoost_R2": xgb["R2"]
    })

regression_baseline_comparison = pd.DataFrame(
    comparison_records
)

print("\n=== XGBOOST VS ZERO-RETURN BASELINE ===")

display(
    regression_baseline_comparison.round(5)
)

# Verify prediction coverage
assert len(regression_validation_predictions) == len(
    validation_data
)

assert not regression_validation_predictions.duplicated(
    ["Ticker", "Date", "Horizon_Days"]
).any()

# Save model results and validation predictions
regression_results_df.to_csv(
    RESULTS_DIR / "regression_xgboost_initial_results.csv",
    index=False
)

regression_baseline_comparison.to_csv(
    RESULTS_DIR / "regression_xgboost_baseline_comparison.csv",
    index=False
)

regression_validation_predictions.to_csv(
    RESULTS_DIR / "regression_xgboost_validation_predictions.csv",
    index=False
)

print("\nCompleted.")

Malaysia | 1-day return | Training: 18618 | Validation: 4864
Malaysia | 5-day return | Training: 18518 | Validation: 4820
Singapore | 1-day return | Training: 5916 | Validation: 1500
Singapore | 5-day return | Training: 5904 | Validation: 1488

=== INITIAL REGRESSION RESULTS ===


,Country,Horizon_Days,Method,Validation_Rows,MAE_pp,RMSE_pp,R2
0,Malaysia,1,Zero_Return,4864,0.72041,1.12097,-0.00158
1,Malaysia,1,Training_Mean,4864,0.72065,1.12089,-0.00145
2,Malaysia,1,XGBoost,4864,0.72604,1.12145,-0.00245
3,Malaysia,5,Training_Mean,4820,1.63787,2.43549,-0.00716
4,Malaysia,5,Zero_Return,4820,1.63788,2.43621,-0.00775
5,Malaysia,5,XGBoost,4820,1.63948,2.43708,-0.00847
6,Singapore,1,Zero_Return,1500,0.68593,0.95726,-0.00406
7,Singapore,1,Training_Mean,1500,0.68623,0.95602,-0.00145
8,Singapore,1,XGBoost,1500,0.69837,0.96805,-0.02682
9,Singapore,5,Training_Mean,1488,1.64563,2.25808,-0.00543



=== XGBOOST VS ZERO-RETURN BASELINE ===


,Country,Horizon_Days,XGBoost_MAE_pp,Zero_MAE_pp,Mean_MAE_pp,MAE_Improvement_vs_Zero_pct,XGBoost_RMSE_pp,Zero_RMSE_pp,RMSE_Improvement_vs_Zero_pct,XGBoost_R2
0,Malaysia,1,0.72604,0.72041,0.72065,-0.78157,1.12145,1.12097,-0.04340,-0.00245
1,Malaysia,5,1.63948,1.63788,1.63787,-0.09781,2.43708,2.43621,-0.03587,-0.00847
2,Singapore,1,0.69837,0.68593,0.68623,-1.81232,0.96805,0.95726,-1.12742,-0.02682
3,Singapore,5,1.73761,1.65349,1.64563,-5.08775,2.36482,2.27110,-4.12660,-0.10274



Completed.


**Analyse regression performance by year and bank**

In [5]:
diagnostic_data = regression_validation_predictions.copy()
diagnostic_data["Year"] = diagnostic_data["Date"].dt.year

def evaluate_regression_group(group):

    actual = group["Target_Return"].to_numpy() * 100
    predicted = group["XGBoost_Predicted_Return"].to_numpy() * 100
    mean_baseline = group["Mean_Predicted_Return"].to_numpy() * 100
    zero_baseline = np.zeros(len(group))

    xgb_mae = mean_absolute_error(actual, predicted)
    zero_mae = mean_absolute_error(actual, zero_baseline)
    mean_mae = mean_absolute_error(actual, mean_baseline)

    xgb_rmse = np.sqrt(mean_squared_error(actual, predicted))
    zero_rmse = np.sqrt(mean_squared_error(actual, zero_baseline))

    correlation = np.nan

    if (
        len(group) > 1
        and np.std(actual) > 1e-12
        and np.std(predicted) > 1e-12
    ):
        correlation = np.corrcoef(actual, predicted)[0, 1]

    return {
        "Observations": len(group),
        "XGBoost_MAE_pp": xgb_mae,
        "Zero_MAE_pp": zero_mae,
        "Mean_MAE_pp": mean_mae,
        "MAE_Difference_vs_Zero_pp": xgb_mae - zero_mae,
        "XGBoost_RMSE_pp": xgb_rmse,
        "Zero_RMSE_pp": zero_rmse,
        "R2": r2_score(actual, predicted),
        "Prediction_Correlation": correlation,
        "Actual_Mean_pp": actual.mean(),
        "Predicted_Mean_pp": predicted.mean(),
        "Actual_Std_pp": actual.std(),
        "Predicted_Std_pp": predicted.std()
    }


# Evaluate performance separately for 2023 and 2024
yearly_records = []

for (country, horizon, year), group in diagnostic_data.groupby(
    ["Country", "Horizon_Days", "Year"]
):
    yearly_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "Year": year,
        **evaluate_regression_group(group)
    })

regression_yearly_diagnostics = pd.DataFrame(yearly_records)

print("=== REGRESSION PERFORMANCE BY YEAR ===")

display(
    regression_yearly_diagnostics[
        [
            "Country",
            "Horizon_Days",
            "Year",
            "Observations",
            "XGBoost_MAE_pp",
            "Zero_MAE_pp",
            "MAE_Difference_vs_Zero_pp",
            "R2",
            "Prediction_Correlation"
        ]
    ].round(5)
)


# Evaluate performance for individual banks
bank_records = []

for (country, horizon, ticker), group in diagnostic_data.groupby(
    ["Country", "Horizon_Days", "Ticker"]
):
    bank_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "Ticker": ticker,
        **evaluate_regression_group(group)
    })

regression_bank_diagnostics = pd.DataFrame(bank_records)

print("\n=== REGRESSION PERFORMANCE BY BANK ===")

display(
    regression_bank_diagnostics[
        [
            "Country",
            "Horizon_Days",
            "Ticker",
            "Observations",
            "XGBoost_MAE_pp",
            "Zero_MAE_pp",
            "MAE_Difference_vs_Zero_pp",
            "R2"
        ]
    ].round(5)
)


# Examine whether predicted returns vary as much as actual returns
prediction_distribution_records = []

for (country, horizon), group in diagnostic_data.groupby(
    ["Country", "Horizon_Days"]
):
    result = evaluate_regression_group(group)

    prediction_distribution_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "Actual_Mean_pp": result["Actual_Mean_pp"],
        "Predicted_Mean_pp": result["Predicted_Mean_pp"],
        "Actual_Std_pp": result["Actual_Std_pp"],
        "Predicted_Std_pp": result["Predicted_Std_pp"],
        "Prediction_Correlation": result["Prediction_Correlation"]
    })

regression_prediction_distribution = pd.DataFrame(
    prediction_distribution_records
)

print("\n=== ACTUAL VS PREDICTED RETURN DISTRIBUTIONS ===")

display(regression_prediction_distribution.round(5))


# Count banks where XGBoost improves upon the zero baseline
bank_improvement_summary = (
    regression_bank_diagnostics
    .assign(
        XGBoost_Better=lambda df:
        df["MAE_Difference_vs_Zero_pp"] < 0
    )
    .groupby(["Country", "Horizon_Days"])
    .agg(
        Banks=("Ticker", "nunique"),
        Banks_XGBoost_Better=("XGBoost_Better", "sum")
    )
    .reset_index()
)

print("\n=== BANK-LEVEL IMPROVEMENT SUMMARY ===")

display(bank_improvement_summary)


# Save diagnostic results
regression_yearly_diagnostics.to_csv(
    RESULTS_DIR / "regression_yearly_diagnostics.csv",
    index=False
)

regression_bank_diagnostics.to_csv(
    RESULTS_DIR / "regression_bank_diagnostics.csv",
    index=False
)

regression_prediction_distribution.to_csv(
    RESULTS_DIR / "regression_prediction_distribution.csv",
    index=False
)

bank_improvement_summary.to_csv(
    RESULTS_DIR / "regression_bank_improvement_summary.csv",
    index=False
)

print("\nCompleted.")

=== REGRESSION PERFORMANCE BY YEAR ===


,Country,Horizon_Days,Year,Observations,XGBoost_MAE_pp,Zero_MAE_pp,MAE_Difference_vs_Zero_pp,R2,Prediction_Correlation
0,Malaysia,1,2023,2440,0.67136,0.66364,0.00772,-0.00551,0.02428
1,Malaysia,1,2024,2424,0.78107,0.77755,0.00353,-0.00313,0.06915
2,Malaysia,5,2023,2440,1.57732,1.57590,0.00142,-0.00573,0.04648
3,Malaysia,5,2024,2380,1.70320,1.70141,0.00178,-0.02288,0.08019
4,Singapore,1,2023,747,0.68305,0.67291,0.01013,-0.02578,0.01641
5,Singapore,1,2024,753,0.71356,0.69885,0.01471,-0.03491,0.04107
6,Singapore,5,2023,747,1.66396,1.55832,0.10565,-0.09338,0.02193
7,Singapore,5,2024,741,1.81186,1.74943,0.06243,-0.15034,0.01324



=== REGRESSION PERFORMANCE BY BANK ===


,Country,Horizon_Days,Ticker,Observations,XGBoost_MAE_pp,Zero_MAE_pp,MAE_Difference_vs_Zero_pp,R2
0,Malaysia,1,1015.KL,490,0.83112,0.83512,-0.00399,0.01209
1,Malaysia,1,1023.KL,490,0.70419,0.69532,0.00887,-0.01352
2,Malaysia,1,1066.KL,490,0.50690,0.50629,0.00061,0.00455
3,Malaysia,1,1155.KL,490,0.42941,0.42749,0.00192,0.00897
4,Malaysia,1,1171.KL,490,1.04885,1.03749,0.01136,-0.00618
5,Malaysia,1,1295.KL,454,0.61571,0.61245,0.00326,-0.01050
6,Malaysia,1,2488.KL,490,0.76604,0.76357,0.00247,0.00383
7,Malaysia,1,5185.KL,490,0.97761,0.95882,0.01880,-0.02054
8,Malaysia,1,5258.KL,490,0.88309,0.87642,0.00668,0.00894
9,Malaysia,1,5819.KL,490,0.48933,0.48317,0.00616,-0.00112



=== ACTUAL VS PREDICTED RETURN DISTRIBUTIONS ===


,Country,Horizon_Days,Actual_Mean_pp,Predicted_Mean_pp,Actual_Std_pp,Predicted_Std_pp,Prediction_Correlation
0,Malaysia,1,0.04455,-0.00183,1.12008,0.12007,0.05016
1,Malaysia,5,0.21364,0.00877,2.42682,0.34980,0.06740
2,Singapore,1,0.06083,0.01286,0.95533,0.17408,0.02443
3,Singapore,5,0.29419,0.06752,2.25197,0.66043,-0.01125



=== BANK-LEVEL IMPROVEMENT SUMMARY ===


,Country,Horizon_Days,Banks,Banks_XGBoost_Better
0,Malaysia,1,10,1
1,Malaysia,5,10,4
2,Singapore,1,3,0
3,Singapore,5,3,0



Completed.


**Evaluate Ridge regression with historical parameter selection**

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

RIDGE_ALPHAS = [1.0, 10.0, 100.0, 1000.0]
INTERNAL_TRAIN_END = pd.Timestamp("2020-12-31")
INTERNAL_VALIDATION_START = pd.Timestamp("2021-01-01")

ridge_selection_records = []
ridge_result_records = []
ridge_prediction_records = []
ridge_models = {}

for country in ["Malaysia", "Singapore"]:

    for horizon in [1, 5]:

        # Select the original training and validation observations
        train = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Train_2015_2022")
        ].copy()

        valid = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Validation_2023_2024")
        ].copy()

        # Define an internal chronological training split
        internal_train = train.loc[
            train["Target_End_Date"] <= INTERNAL_TRAIN_END
        ].copy()

        internal_valid = train.loc[
            (train["Date"] >= INTERNAL_VALIDATION_START)
            & (train["Target_End_Date"] <= TRAIN_END)
        ].copy()

        assert len(internal_train) > 0
        assert len(internal_valid) > 0
        assert (
            internal_train["Target_End_Date"].max()
            < internal_valid["Date"].min()
        )

        X_internal_train = internal_train[feature_columns]
        y_internal_train = (
            internal_train["Target_Return"].to_numpy() * 100
        )

        X_internal_valid = internal_valid[feature_columns]
        y_internal_valid = (
            internal_valid["Target_Return"].to_numpy() * 100
        )

        # Select regularisation using 2021–2022 only
        for alpha in RIDGE_ALPHAS:

            candidate = make_pipeline(
                StandardScaler(),
                Ridge(alpha=alpha)
            )

            candidate.fit(X_internal_train, y_internal_train)

            internal_predictions = candidate.predict(
                X_internal_valid
            )

            ridge_selection_records.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Alpha": alpha,
                "Internal_Train_Rows": len(internal_train),
                "Internal_Validation_Rows": len(internal_valid),
                "Internal_MAE_pp": mean_absolute_error(
                    y_internal_valid,
                    internal_predictions
                ),
                "Internal_RMSE_pp": np.sqrt(
                    mean_squared_error(
                        y_internal_valid,
                        internal_predictions
                    )
                )
            })

        # Retrieve the alpha selected from historical data
        selection = pd.DataFrame(ridge_selection_records)

        best_row = (
            selection.loc[
                (selection["Country"] == country)
                & (selection["Horizon_Days"] == horizon)
            ]
            .sort_values(["Internal_MAE_pp", "Alpha"])
            .iloc[0]
        )

        best_alpha = float(best_row["Alpha"])

        # Refit using the full 2015–2022 training period
        final_model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=best_alpha)
        )

        final_model.fit(
            train[feature_columns],
            train["Target_Return"].to_numpy() * 100
        )

        ridge_models[(country, horizon)] = final_model

        # Evaluate the selected model on 2023–2024
        y_valid = valid["Target_Return"].to_numpy() * 100

        predictions = final_model.predict(
            valid[feature_columns]
        )

        ridge_result_records.append({
            "Country": country,
            "Horizon_Days": horizon,
            "Selected_Alpha": best_alpha,
            "Training_Rows": len(train),
            "Validation_Rows": len(valid),
            "MAE_pp": mean_absolute_error(
                y_valid,
                predictions
            ),
            "RMSE_pp": np.sqrt(
                mean_squared_error(
                    y_valid,
                    predictions
                )
            ),
            "R2": r2_score(y_valid, predictions),
            "Prediction_Correlation": (
                np.corrcoef(y_valid, predictions)[0, 1]
                if np.std(predictions) > 1e-12
                else np.nan
            )
        })

        prediction_output = valid[
            [
                "Ticker",
                "Country",
                "Date",
                "Horizon_Days",
                "Target_End_Date",
                "Target_Return"
            ]
        ].copy()

        prediction_output["Ridge_Predicted_Return"] = (
            predictions / 100
        )

        ridge_prediction_records.append(prediction_output)

        print(
            f"{country} | {horizon}-day return | "
            f"Selected alpha: {best_alpha:g}"
        )

# Combine results
ridge_selection_results = pd.DataFrame(
    ridge_selection_records
)

ridge_results_df = pd.DataFrame(
    ridge_result_records
)

ridge_validation_predictions = pd.concat(
    ridge_prediction_records,
    ignore_index=True
)

print("\n=== INTERNAL REGULARISATION SELECTION ===")

display(
    ridge_selection_results.sort_values(
        ["Country", "Horizon_Days", "Alpha"]
    ).round(5)
)

print("\n=== RIDGE VALIDATION RESULTS ===")

display(ridge_results_df.round(5))

# Compare Ridge with the existing regression methods
ridge_comparison = pd.concat(
    [
        regression_results_df[
            [
                "Country",
                "Horizon_Days",
                "Method",
                "Validation_Rows",
                "MAE_pp",
                "RMSE_pp",
                "R2"
            ]
        ],
        ridge_results_df[
            [
                "Country",
                "Horizon_Days",
                "Validation_Rows",
                "MAE_pp",
                "RMSE_pp",
                "R2"
            ]
        ].assign(Method="Ridge")
    ],
    ignore_index=True
)

ridge_comparison = ridge_comparison.sort_values(
    ["Country", "Horizon_Days", "MAE_pp"]
).reset_index(drop=True)

print("\n=== REGRESSION MODEL COMPARISON ===")

display(ridge_comparison.round(5))

# Validate observation coverage
assert len(ridge_validation_predictions) == len(
    regression_validation_predictions
)

# Save results
ridge_selection_results.to_csv(
    RESULTS_DIR / "regression_ridge_internal_selection.csv",
    index=False
)

ridge_results_df.to_csv(
    RESULTS_DIR / "regression_ridge_results.csv",
    index=False
)

ridge_validation_predictions.to_csv(
    RESULTS_DIR / "regression_ridge_validation_predictions.csv",
    index=False
)

ridge_comparison.to_csv(
    RESULTS_DIR / "regression_ridge_model_comparison.csv",
    index=False
)

print("\nCompleted.")

Malaysia | 1-day return | Selected alpha: 1000
Malaysia | 5-day return | Selected alpha: 1000
Singapore | 1-day return | Selected alpha: 1000
Singapore | 5-day return | Selected alpha: 1000

=== INTERNAL REGULARISATION SELECTION ===


,Country,Horizon_Days,Alpha,Internal_Train_Rows,Internal_Validation_Rows,Internal_MAE_pp,Internal_RMSE_pp
0,Malaysia,1,1.0,13852,4756,0.88128,1.27888
1,Malaysia,1,10.0,13852,4756,0.88083,1.27837
2,Malaysia,1,100.0,13852,4756,0.88034,1.27786
3,Malaysia,1,1000.0,13852,4756,0.87942,1.27777
4,Malaysia,5,1.0,13764,4704,1.93019,2.68773
5,Malaysia,5,10.0,13764,4704,1.92882,2.68552
6,Malaysia,5,100.0,13764,4704,1.92729,2.68291
7,Malaysia,5,1000.0,13764,4704,1.92330,2.67976
8,Singapore,1,1.0,4407,1506,0.77948,1.06424
9,Singapore,1,10.0,4407,1506,0.77782,1.06236



=== RIDGE VALIDATION RESULTS ===


,Country,Horizon_Days,Selected_Alpha,Training_Rows,Validation_Rows,MAE_pp,RMSE_pp,R2,Prediction_Correlation
0,Malaysia,1,1000.0,18618,4864,0.72747,1.12211,-0.00363,0.02164
1,Malaysia,5,1000.0,18518,4820,1.65526,2.44472,-0.01481,0.00807
2,Singapore,1,1000.0,5916,1500,0.68897,0.95695,-0.00341,0.05015
3,Singapore,5,1000.0,5904,1488,1.68961,2.29960,-0.04275,0.01133



=== REGRESSION MODEL COMPARISON ===


,Country,Horizon_Days,Method,Validation_Rows,MAE_pp,RMSE_pp,R2
0,Malaysia,1,Zero_Return,4864,0.72041,1.12097,-0.00158
1,Malaysia,1,Training_Mean,4864,0.72065,1.12089,-0.00145
2,Malaysia,1,XGBoost,4864,0.72604,1.12145,-0.00245
3,Malaysia,1,Ridge,4864,0.72747,1.12211,-0.00363
4,Malaysia,5,Training_Mean,4820,1.63787,2.43549,-0.00716
5,Malaysia,5,Zero_Return,4820,1.63788,2.43621,-0.00775
6,Malaysia,5,XGBoost,4820,1.63948,2.43708,-0.00847
7,Malaysia,5,Ridge,4820,1.65526,2.44472,-0.01481
8,Singapore,1,Zero_Return,1500,0.68593,0.95726,-0.00406
9,Singapore,1,Training_Mean,1500,0.68623,0.95602,-0.00145



Completed.


**Evaluate stronger Ridge regularisation and median baselines**

In [7]:
EXTENDED_RIDGE_ALPHAS = [
    1.0, 10.0, 100.0, 1000.0,
    10000.0, 100000.0, 1000000.0
]

extended_ridge_selection = []
extended_ridge_results = []
extended_ridge_predictions = []
extended_ridge_models = {}

for country in ["Malaysia", "Singapore"]:

    for horizon in [1, 5]:

        # Select training and validation observations
        train = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Train_2015_2022")
        ].copy()

        valid = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Validation_2023_2024")
        ].copy()

        # Prepare historical parameter-selection periods
        internal_train = train.loc[
            train["Target_End_Date"] <= INTERNAL_TRAIN_END
        ]

        internal_valid = train.loc[
            (train["Date"] >= INTERNAL_VALIDATION_START)
            & (train["Target_End_Date"] <= TRAIN_END)
        ]

        X_internal_train = internal_train[feature_columns]
        y_internal_train = (
            internal_train["Target_Return"].to_numpy() * 100
        )

        X_internal_valid = internal_valid[feature_columns]
        y_internal_valid = (
            internal_valid["Target_Return"].to_numpy() * 100
        )

        assert (
            internal_train["Target_End_Date"].max()
            < internal_valid["Date"].min()
        )

        # Evaluate Ridge strengths using historical validation
        for alpha in EXTENDED_RIDGE_ALPHAS:

            candidate = make_pipeline(
                StandardScaler(),
                Ridge(alpha=alpha)
            )

            candidate.fit(
                X_internal_train,
                y_internal_train
            )

            predicted = candidate.predict(X_internal_valid)

            extended_ridge_selection.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Alpha": alpha,
                "Internal_MAE_pp": mean_absolute_error(
                    y_internal_valid, predicted
                ),
                "Internal_RMSE_pp": np.sqrt(
                    mean_squared_error(
                        y_internal_valid, predicted
                    )
                )
            })

        # Select alpha using internal MAE
        selection_subset = pd.DataFrame(
            extended_ridge_selection
        )

        selection_subset = selection_subset.loc[
            (selection_subset["Country"] == country)
            & (selection_subset["Horizon_Days"] == horizon)
        ]

        best_alpha = float(
            selection_subset.sort_values(
                ["Internal_MAE_pp", "Alpha"]
            ).iloc[0]["Alpha"]
        )

        # Refit Ridge using the full historical training period
        final_model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=best_alpha)
        )

        y_train = train["Target_Return"].to_numpy() * 100
        y_valid = valid["Target_Return"].to_numpy() * 100

        final_model.fit(
            train[feature_columns],
            y_train
        )

        ridge_pred = final_model.predict(
            valid[feature_columns]
        )

        extended_ridge_models[(country, horizon)] = final_model

        # Calculate constant forecasts from training observations
        median_pred = np.full(
            len(valid),
            np.median(y_train)
        )

        zero_pred = np.zeros(len(valid))

        mean_pred = np.full(
            len(valid),
            np.mean(y_train)
        )

        # Compare predictions on identical validation observations
        for method, predictions in {
            "Extended_Ridge": ridge_pred,
            "Training_Median": median_pred,
            "Zero_Return": zero_pred,
            "Training_Mean": mean_pred
        }.items():

            extended_ridge_results.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Method": method,
                "Selected_Alpha": (
                    best_alpha if method == "Extended_Ridge"
                    else np.nan
                ),
                "Validation_Rows": len(valid),
                "MAE_pp": mean_absolute_error(
                    y_valid, predictions
                ),
                "RMSE_pp": np.sqrt(
                    mean_squared_error(
                        y_valid, predictions
                    )
                ),
                "R2": r2_score(
                    y_valid, predictions
                )
            })

        prediction_output = valid[
            [
                "Ticker", "Country", "Date",
                "Horizon_Days", "Target_End_Date",
                "Target_Return"
            ]
        ].copy()

        prediction_output["Extended_Ridge_Return"] = (
            ridge_pred / 100
        )

        prediction_output["Median_Baseline_Return"] = (
            median_pred / 100
        )

        extended_ridge_predictions.append(prediction_output)

        print(
            f"{country} | {horizon}-day return | "
            f"Selected alpha: {best_alpha:g}"
        )

# Combine experiment results
extended_ridge_selection_df = pd.DataFrame(
    extended_ridge_selection
)

extended_ridge_results_df = pd.DataFrame(
    extended_ridge_results
)

extended_ridge_predictions_df = pd.concat(
    extended_ridge_predictions,
    ignore_index=True
)

print("\n=== EXTENDED RIDGE PARAMETER SELECTION ===")

display(
    extended_ridge_selection_df.sort_values(
        ["Country", "Horizon_Days", "Alpha"]
    ).round(5)
)

print("\n=== EXTENDED RIDGE AND CONSTANT BASELINES ===")

display(
    extended_ridge_results_df.sort_values(
        ["Country", "Horizon_Days", "MAE_pp"]
    ).round(5)
)

# Compare the new Ridge model against the previous XGBoost
xgb_reference = regression_results_df.loc[
    regression_results_df["Method"] == "XGBoost",
    [
        "Country", "Horizon_Days", "Method",
        "Validation_Rows", "MAE_pp", "RMSE_pp", "R2"
    ]
]

extended_comparison = pd.concat(
    [
        extended_ridge_results_df[
            [
                "Country", "Horizon_Days", "Method",
                "Validation_Rows", "MAE_pp", "RMSE_pp", "R2"
            ]
        ],
        xgb_reference
    ],
    ignore_index=True
)

print("\n=== REGRESSION MODEL COMPARISON ===")

display(
    extended_comparison.sort_values(
        ["Country", "Horizon_Days", "MAE_pp"]
    ).round(5)
)

# Save results
extended_ridge_selection_df.to_csv(
    RESULTS_DIR / "regression_ridge_extended_selection.csv",
    index=False
)

extended_ridge_results_df.to_csv(
    RESULTS_DIR / "regression_ridge_extended_results.csv",
    index=False
)

extended_ridge_predictions_df.to_csv(
    RESULTS_DIR / "regression_ridge_extended_predictions.csv",
    index=False
)

extended_comparison.to_csv(
    RESULTS_DIR / "regression_extended_model_comparison.csv",
    index=False
)

print("\nCompleted.")

Malaysia | 1-day return | Selected alpha: 1e+06
Malaysia | 5-day return | Selected alpha: 100000
Singapore | 1-day return | Selected alpha: 1000
Singapore | 5-day return | Selected alpha: 10000

=== EXTENDED RIDGE PARAMETER SELECTION ===


,Country,Horizon_Days,Alpha,Internal_MAE_pp,Internal_RMSE_pp
0,Malaysia,1,1.0,0.88128,1.27888
1,Malaysia,1,10.0,0.88083,1.27837
2,Malaysia,1,100.0,0.88034,1.27786
3,Malaysia,1,1000.0,0.87942,1.27777
4,Malaysia,1,10000.0,0.87584,1.27672
5,Malaysia,1,100000.0,0.87300,1.27673
6,Malaysia,1,1000000.0,0.87288,1.27724
7,Malaysia,5,1.0,1.93019,2.68773
8,Malaysia,5,10.0,1.92882,2.68552
9,Malaysia,5,100.0,1.92729,2.68291



=== EXTENDED RIDGE AND CONSTANT BASELINES ===


,Country,Horizon_Days,Method,Selected_Alpha,Validation_Rows,MAE_pp,RMSE_pp,R2
1,Malaysia,1,Training_Median,NaN,4864,0.72041,1.12097,-0.00158
2,Malaysia,1,Zero_Return,NaN,4864,0.72041,1.12097,-0.00158
0,Malaysia,1,Extended_Ridge,1000000.0,4864,0.72057,1.12075,-0.00119
3,Malaysia,1,Training_Mean,NaN,4864,0.72065,1.12089,-0.00145
4,Malaysia,5,Extended_Ridge,100000.0,4820,1.63778,2.43390,-0.00584
7,Malaysia,5,Training_Mean,NaN,4820,1.63787,2.43549,-0.00716
5,Malaysia,5,Training_Median,NaN,4820,1.63788,2.43621,-0.00775
6,Malaysia,5,Zero_Return,NaN,4820,1.63788,2.43621,-0.00775
9,Singapore,1,Training_Median,NaN,1500,0.68593,0.95726,-0.00406
10,Singapore,1,Zero_Return,NaN,1500,0.68593,0.95726,-0.00406



=== REGRESSION MODEL COMPARISON ===


,Country,Horizon_Days,Method,Validation_Rows,MAE_pp,RMSE_pp,R2
1,Malaysia,1,Training_Median,4864,0.72041,1.12097,-0.00158
2,Malaysia,1,Zero_Return,4864,0.72041,1.12097,-0.00158
0,Malaysia,1,Extended_Ridge,4864,0.72057,1.12075,-0.00119
3,Malaysia,1,Training_Mean,4864,0.72065,1.12089,-0.00145
16,Malaysia,1,XGBoost,4864,0.72604,1.12145,-0.00245
4,Malaysia,5,Extended_Ridge,4820,1.63778,2.43390,-0.00584
7,Malaysia,5,Training_Mean,4820,1.63787,2.43549,-0.00716
5,Malaysia,5,Training_Median,4820,1.63788,2.43621,-0.00775
6,Malaysia,5,Zero_Return,4820,1.63788,2.43621,-0.00775
17,Malaysia,5,XGBoost,4820,1.63948,2.43708,-0.00847



Completed.


**Tune conservative XGBoost regression configurations**

In [8]:
xgb_candidates = [
    {
        "Name": "Original",
        "Overrides": {}
    },
    {
        "Name": "Stump_Light",
        "Overrides": {
            "n_estimators": 60,
            "learning_rate": 0.03,
            "max_depth": 1,
            "min_child_weight": 10,
            "reg_lambda": 10,
            "subsample": 1.0,
            "colsample_bytree": 1.0
        }
    },
    {
        "Name": "Stump_Strong",
        "Overrides": {
            "n_estimators": 120,
            "learning_rate": 0.02,
            "max_depth": 1,
            "min_child_weight": 30,
            "reg_lambda": 100,
            "subsample": 1.0,
            "colsample_bytree": 1.0
        }
    },
    {
        "Name": "Depth2_Conservative",
        "Overrides": {
            "n_estimators": 80,
            "learning_rate": 0.03,
            "max_depth": 2,
            "min_child_weight": 15,
            "reg_lambda": 30,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        }
    },
    {
        "Name": "Depth2_Strong",
        "Overrides": {
            "n_estimators": 150,
            "learning_rate": 0.02,
            "max_depth": 2,
            "min_child_weight": 30,
            "reg_lambda": 100,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        }
    },
    {
        "Name": "Depth3_Regularised",
        "Overrides": {
            "n_estimators": 100,
            "learning_rate": 0.03,
            "max_depth": 3,
            "min_child_weight": 20,
            "reg_lambda": 50,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    }
]

xgb_tuning_records = []
xgb_tuned_result_records = []
xgb_tuned_prediction_records = []
xgb_tuned_models = {}

for country in ["Malaysia", "Singapore"]:

    for horizon in [1, 5]:

        train = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Train_2015_2022")
        ].copy()

        valid = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (regression_model_data["Split"] == "Validation_2023_2024")
        ].copy()

        # Prepare chronological internal training and validation
        internal_train = train.loc[
            train["Target_End_Date"] <= INTERNAL_TRAIN_END
        ]

        internal_valid = train.loc[
            (train["Date"] >= INTERNAL_VALIDATION_START)
            & (train["Target_End_Date"] <= TRAIN_END)
        ]

        assert (
            internal_train["Target_End_Date"].max()
            < internal_valid["Date"].min()
        )

        X_internal_train = internal_train[feature_columns]
        y_internal_train = (
            internal_train["Target_Return"].to_numpy() * 100
        )

        X_internal_valid = internal_valid[feature_columns]
        y_internal_valid = (
            internal_valid["Target_Return"].to_numpy() * 100
        )

        # Evaluate each predefined XGBoost configuration
        for candidate in xgb_candidates:

            parameters = {
                **regression_params,
                **candidate["Overrides"]
            }

            model = XGBRegressor(**parameters)

            model.fit(
                X_internal_train,
                y_internal_train,
                verbose=False
            )

            predictions = model.predict(X_internal_valid)

            xgb_tuning_records.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Configuration": candidate["Name"],
                "Internal_Train_Rows": len(internal_train),
                "Internal_Validation_Rows": len(internal_valid),
                "Internal_MAE_pp": mean_absolute_error(
                    y_internal_valid,
                    predictions
                ),
                "Internal_RMSE_pp": np.sqrt(
                    mean_squared_error(
                        y_internal_valid,
                        predictions
                    )
                )
            })

        # Select the configuration using internal validation MAE
        selection = pd.DataFrame(xgb_tuning_records)

        selection = selection.loc[
            (selection["Country"] == country)
            & (selection["Horizon_Days"] == horizon)
        ].sort_values(
            ["Internal_MAE_pp", "Configuration"]
        )

        best_name = selection.iloc[0]["Configuration"]

        best_candidate = next(
            candidate for candidate in xgb_candidates
            if candidate["Name"] == best_name
        )

        # Fit selected configuration using all 2015–2022 data
        final_parameters = {
            **regression_params,
            **best_candidate["Overrides"]
        }

        final_model = XGBRegressor(**final_parameters)

        final_model.fit(
            train[feature_columns],
            train["Target_Return"].to_numpy() * 100,
            verbose=False
        )

        xgb_tuned_models[(country, horizon)] = final_model

        # Evaluate on the unchanged 2023–2024 validation sample
        y_valid = valid["Target_Return"].to_numpy() * 100

        predictions = final_model.predict(
            valid[feature_columns]
        )

        xgb_tuned_result_records.append({
            "Country": country,
            "Horizon_Days": horizon,
            "Selected_Configuration": best_name,
            "Training_Rows": len(train),
            "Validation_Rows": len(valid),
            "MAE_pp": mean_absolute_error(
                y_valid,
                predictions
            ),
            "RMSE_pp": np.sqrt(
                mean_squared_error(
                    y_valid,
                    predictions
                )
            ),
            "R2": r2_score(y_valid, predictions),
            "Prediction_Correlation": (
                np.corrcoef(y_valid, predictions)[0, 1]
                if np.std(predictions) > 1e-12
                else np.nan
            )
        })

        prediction_output = valid[
            [
                "Ticker",
                "Country",
                "Date",
                "Horizon_Days",
                "Target_End_Date",
                "Target_Return"
            ]
        ].copy()

        prediction_output["Tuned_XGBoost_Return"] = (
            predictions / 100
        )

        xgb_tuned_prediction_records.append(
            prediction_output
        )

        print(
            f"{country} | {horizon}-day return | "
            f"Selected: {best_name}"
        )

# Combine tuning results and validation predictions
xgb_tuning_results = pd.DataFrame(xgb_tuning_records)

xgb_tuned_results = pd.DataFrame(
    xgb_tuned_result_records
)

xgb_tuned_predictions = pd.concat(
    xgb_tuned_prediction_records,
    ignore_index=True
)

assert len(xgb_tuned_predictions) == len(
    regression_validation_predictions
)

print("\n=== INTERNAL XGBOOST HYPERPARAMETER RESULTS ===")

display(
    xgb_tuning_results.sort_values(
        ["Country", "Horizon_Days", "Internal_MAE_pp"]
    ).round(5)
)

print("\n=== TUNED XGBOOST VALIDATION RESULTS ===")

display(xgb_tuned_results.round(5))

# Compare tuned XGBoost with existing regression models
tuned_xgb_comparison = pd.concat(
    [
        extended_comparison,
        xgb_tuned_results[
            [
                "Country",
                "Horizon_Days",
                "Validation_Rows",
                "MAE_pp",
                "RMSE_pp",
                "R2"
            ]
        ].assign(Method="Tuned_XGBoost")
    ],
    ignore_index=True
)

tuned_xgb_comparison = tuned_xgb_comparison.sort_values(
    ["Country", "Horizon_Days", "MAE_pp"]
).reset_index(drop=True)

print("\n=== REGRESSION MODEL COMPARISON ===")

display(tuned_xgb_comparison.round(5))

# Save experimental results
xgb_tuning_results.to_csv(
    RESULTS_DIR / "regression_xgboost_tuning_selection.csv",
    index=False
)

xgb_tuned_results.to_csv(
    RESULTS_DIR / "regression_xgboost_tuned_results.csv",
    index=False
)

xgb_tuned_predictions.to_csv(
    RESULTS_DIR / "regression_xgboost_tuned_predictions.csv",
    index=False
)

tuned_xgb_comparison.to_csv(
    RESULTS_DIR / "regression_xgboost_tuned_comparison.csv",
    index=False
)

print("\nCompleted.")

Malaysia | 1-day return | Selected: Stump_Light
Malaysia | 5-day return | Selected: Stump_Light
Singapore | 1-day return | Selected: Depth2_Strong
Singapore | 5-day return | Selected: Depth3_Regularised

=== INTERNAL XGBOOST HYPERPARAMETER RESULTS ===


,Country,Horizon_Days,Configuration,Internal_Train_Rows,Internal_Validation_Rows,Internal_MAE_pp,Internal_RMSE_pp
1,Malaysia,1,Stump_Light,13852,4756,0.87315,1.27792
2,Malaysia,1,Stump_Strong,13852,4756,0.87347,1.27736
4,Malaysia,1,Depth2_Strong,13852,4756,0.87380,1.27675
3,Malaysia,1,Depth2_Conservative,13852,4756,0.87394,1.27706
5,Malaysia,1,Depth3_Regularised,13852,4756,0.87448,1.27755
0,Malaysia,1,Original,13852,4756,0.88150,1.28470
7,Malaysia,5,Stump_Light,13764,4704,1.90495,2.66353
8,Malaysia,5,Stump_Strong,13764,4704,1.90760,2.66639
9,Malaysia,5,Depth2_Conservative,13764,4704,1.91182,2.67605
10,Malaysia,5,Depth2_Strong,13764,4704,1.91293,2.67784



=== TUNED XGBOOST VALIDATION RESULTS ===


,Country,Horizon_Days,Selected_Configuration,Training_Rows,Validation_Rows,MAE_pp,RMSE_pp,R2,Prediction_Correlation
0,Malaysia,1,Stump_Light,18618,4864,0.72153,1.12032,-0.00042,0.04424
1,Malaysia,5,Stump_Light,18518,4820,1.63774,2.43464,-0.00646,0.05550
2,Singapore,1,Depth2_Strong,5916,1500,0.68905,0.95855,-0.00675,0.01495
3,Singapore,5,Depth3_Regularised,5904,1488,1.68387,2.30482,-0.04748,-0.00096



=== REGRESSION MODEL COMPARISON ===


,Country,Horizon_Days,Method,Validation_Rows,MAE_pp,RMSE_pp,R2
0,Malaysia,1,Training_Median,4864,0.72041,1.12097,-0.00158
1,Malaysia,1,Zero_Return,4864,0.72041,1.12097,-0.00158
2,Malaysia,1,Extended_Ridge,4864,0.72057,1.12075,-0.00119
3,Malaysia,1,Training_Mean,4864,0.72065,1.12089,-0.00145
4,Malaysia,1,Tuned_XGBoost,4864,0.72153,1.12032,-0.00042
5,Malaysia,1,XGBoost,4864,0.72604,1.12145,-0.00245
6,Malaysia,5,Tuned_XGBoost,4820,1.63774,2.43464,-0.00646
7,Malaysia,5,Extended_Ridge,4820,1.63778,2.43390,-0.00584
8,Malaysia,5,Training_Mean,4820,1.63787,2.43549,-0.00716
9,Malaysia,5,Training_Median,4820,1.63788,2.43621,-0.00775



Completed.


**Evaluate weekly expanding XGBoost regression**

In [9]:
walkforward_predictions = []
walkforward_training_log = []

for country in ["Malaysia", "Singapore"]:

    for horizon in [1, 5]:

        target_col = f"Target_Return_{horizon}D"
        end_col = f"Target_End_Date_{horizon}D"

        # Select complete historical observations
        country_history = regression_data.loc[
            (regression_data["Country"] == country)
            & regression_data["Feature_Complete"]
            & regression_data[target_col].notna()
            & regression_data[end_col].notna()
        ].copy()

        country_history = country_history.loc[
            np.isfinite(country_history[target_col])
        ]

        # Select the unchanged validation observations
        country_valid = regression_model_data.loc[
            (regression_model_data["Country"] == country)
            & (regression_model_data["Horizon_Days"] == horizon)
            & (
                regression_model_data["Split"]
                == "Validation_2023_2024"
            )
        ].copy()

        country_valid["Week_Start"] = (
            country_valid["Date"]
            - pd.to_timedelta(
                country_valid["Date"].dt.weekday,
                unit="D"
            )
        )

        # Reuse the hyperparameters selected in Cell 8
        selected_name = xgb_tuned_results.loc[
            (xgb_tuned_results["Country"] == country)
            & (xgb_tuned_results["Horizon_Days"] == horizon),
            "Selected_Configuration"
        ].iloc[0]

        selected_candidate = next(
            candidate for candidate in xgb_candidates
            if candidate["Name"] == selected_name
        )

        parameters = {
            **regression_params,
            **selected_candidate["Overrides"]
        }

        weekly_groups = list(
            country_valid.groupby("Week_Start", sort=True)
        )

        print(f"\n=== {country.upper()} | {horizon}-DAY ===")
        print("Selected configuration:", selected_name)
        print("Prediction weeks:", len(weekly_groups))

        for week_number, (week_start, week_data) in enumerate(
            weekly_groups,
            start=1
        ):

            # Include only observations with known outcomes
            train = country_history.loc[
                (country_history["Date"] < week_start)
                & (
                    country_history[end_col]
                    < week_start
                )
            ]

            assert len(train) > 0
            assert train["Date"].max() < week_start
            assert train[end_col].max() < week_start

            X_train = train[feature_columns]
            y_train = train[target_col].to_numpy() * 100

            X_valid = week_data[feature_columns]

            assert np.isfinite(X_train.to_numpy()).all()
            assert np.isfinite(y_train).all()
            assert np.isfinite(X_valid.to_numpy()).all()

            # Retrain once at the beginning of the week
            model = XGBRegressor(**parameters)

            model.fit(
                X_train,
                y_train,
                verbose=False
            )

            predicted = model.predict(X_valid)

            # Calculate baselines from available history
            weekly_mean = float(np.mean(y_train))
            weekly_median = float(np.median(y_train))

            week_predictions = week_data[
                [
                    "Ticker",
                    "Country",
                    "Date",
                    "Horizon_Days",
                    "Target_End_Date",
                    "Target_Return",
                    "Week_Start"
                ]
            ].copy()

            week_predictions["Expanding_XGBoost_Return"] = (
                predicted / 100
            )

            week_predictions["Weekly_Mean_Return"] = (
                weekly_mean / 100
            )

            week_predictions["Weekly_Median_Return"] = (
                weekly_median / 100
            )

            walkforward_predictions.append(week_predictions)

            walkforward_training_log.append({
                "Country": country,
                "Horizon_Days": horizon,
                "Week_Start": week_start,
                "Training_Rows": len(train),
                "Prediction_Rows": len(week_data),
                "Latest_Training_Target_End": (
                    train[end_col].max()
                ),
                "Selected_Configuration": selected_name
            })

            if (
                week_number == 1
                or week_number % 25 == 0
                or week_number == len(weekly_groups)
            ):
                print(
                    f"Week {week_number}/{len(weekly_groups)} | "
                    f"Training rows: {len(train)} | "
                    f"Prediction rows: {len(week_data)}"
                )

# Consolidate predictions and training records
regression_walkforward_predictions = pd.concat(
    walkforward_predictions,
    ignore_index=True
)

regression_walkforward_log = pd.DataFrame(
    walkforward_training_log
)

assert len(regression_walkforward_predictions) == len(
    regression_validation_predictions
)

assert not regression_walkforward_predictions.duplicated(
    ["Ticker", "Date", "Horizon_Days"]
).any()

assert (
    regression_walkforward_log["Latest_Training_Target_End"]
    < regression_walkforward_log["Week_Start"]
).all()

# Match predictions with the original fixed tuned models
fixed_reference = xgb_tuned_predictions[
    [
        "Ticker",
        "Date",
        "Horizon_Days",
        "Target_Return",
        "Tuned_XGBoost_Return"
    ]
].copy()

regression_walkforward_comparison_data = (
    regression_walkforward_predictions.merge(
        fixed_reference,
        on=["Ticker", "Date", "Horizon_Days"],
        how="inner",
        suffixes=("", "_Fixed"),
        validate="one_to_one"
    )
)

assert len(regression_walkforward_comparison_data) == len(
    regression_walkforward_predictions
)

assert np.allclose(
    regression_walkforward_comparison_data["Target_Return"],
    regression_walkforward_comparison_data["Target_Return_Fixed"]
)

# Evaluate the competing prediction methods
walkforward_result_records = []

prediction_methods = {
    "Fixed_Tuned_XGBoost": "Tuned_XGBoost_Return",
    "Weekly_Expanding_XGBoost": "Expanding_XGBoost_Return",
    "Weekly_Training_Mean": "Weekly_Mean_Return",
    "Weekly_Training_Median": "Weekly_Median_Return"
}

for (country, horizon), group in (
    regression_walkforward_comparison_data.groupby(
        ["Country", "Horizon_Days"]
    )
):

    actual = group["Target_Return"].to_numpy() * 100

    for method, column in prediction_methods.items():

        predicted = group[column].to_numpy() * 100

        walkforward_result_records.append({
            "Country": country,
            "Horizon_Days": horizon,
            "Method": method,
            "Validation_Rows": len(group),
            "MAE_pp": mean_absolute_error(
                actual, predicted
            ),
            "RMSE_pp": np.sqrt(
                mean_squared_error(actual, predicted)
            ),
            "R2": r2_score(actual, predicted)
        })

    zero_predictions = np.zeros(len(group))

    walkforward_result_records.append({
        "Country": country,
        "Horizon_Days": horizon,
        "Method": "Zero_Return",
        "Validation_Rows": len(group),
        "MAE_pp": mean_absolute_error(
            actual, zero_predictions
        ),
        "RMSE_pp": np.sqrt(
            mean_squared_error(actual, zero_predictions)
        ),
        "R2": r2_score(actual, zero_predictions)
    })

regression_walkforward_results = pd.DataFrame(
    walkforward_result_records
)

regression_walkforward_results = (
    regression_walkforward_results
    .sort_values(
        ["Country", "Horizon_Days", "MAE_pp"]
    )
    .reset_index(drop=True)
)

print("\n=== WEEKLY EXPANDING REGRESSION COMPARISON ===")

display(regression_walkforward_results.round(5))

print("\n=== WEEKLY TRAINING HISTORY SUMMARY ===")

display(
    regression_walkforward_log.groupby(
        ["Country", "Horizon_Days"]
    ).agg(
        Weeks=("Week_Start", "nunique"),
        Min_Training_Rows=("Training_Rows", "min"),
        Max_Training_Rows=("Training_Rows", "max"),
        Total_Predictions=("Prediction_Rows", "sum")
    )
)

# Save walk-forward experiment results
regression_walkforward_predictions.to_csv(
    RESULTS_DIR / "regression_weekly_expanding_predictions.csv",
    index=False
)

regression_walkforward_log.to_csv(
    RESULTS_DIR / "regression_weekly_expanding_training_log.csv",
    index=False
)

regression_walkforward_results.to_csv(
    RESULTS_DIR / "regression_weekly_expanding_results.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA | 1-DAY ===
Selected configuration: Stump_Light
Prediction weeks: 105
Week 1/105 | Training rows: 18618 | Prediction rows: 40
Week 25/105 | Training rows: 19718 | Prediction rows: 50
Week 50/105 | Training rows: 20918 | Prediction rows: 50
Week 75/105 | Training rows: 22068 | Prediction rows: 40
Week 100/105 | Training rows: 23236 | Prediction rows: 47
Week 105/105 | Training rows: 23472 | Prediction rows: 10

=== MALAYSIA | 5-DAY ===
Selected configuration: Stump_Light
Prediction weeks: 104
Week 1/104 | Training rows: 18518 | Prediction rows: 40
Week 25/104 | Training rows: 19618 | Prediction rows: 50
Week 50/104 | Training rows: 20818 | Prediction rows: 50
Week 75/104 | Training rows: 21968 | Prediction rows: 40
Week 100/104 | Training rows: 23136 | Prediction rows: 47
Week 104/104 | Training rows: 23328 | Prediction rows: 10

=== SINGAPORE | 1-DAY ===
Selected configuration: Depth2_Strong
Prediction weeks: 105
Week 1/105 | Training rows: 5916 | Prediction rows: 12
Week

,Country,Horizon_Days,Method,Validation_Rows,MAE_pp,RMSE_pp,R2
0,Malaysia,1,Weekly_Training_Median,4864,0.72041,1.12097,-0.00158
1,Malaysia,1,Zero_Return,4864,0.72041,1.12097,-0.00158
2,Malaysia,1,Weekly_Training_Mean,4864,0.72094,1.12082,-0.00132
3,Malaysia,1,Weekly_Expanding_XGBoost,4864,0.72151,1.12028,-0.00036
4,Malaysia,1,Fixed_Tuned_XGBoost,4864,0.72153,1.12032,-0.00042
5,Malaysia,5,Fixed_Tuned_XGBoost,4820,1.63774,2.43464,-0.00646
6,Malaysia,5,Weekly_Training_Median,4820,1.63788,2.43621,-0.00775
7,Malaysia,5,Zero_Return,4820,1.63788,2.43621,-0.00775
8,Malaysia,5,Weekly_Training_Mean,4820,1.63802,2.43479,-0.00657
9,Malaysia,5,Weekly_Expanding_XGBoost,4820,1.63835,2.43468,-0.00648



=== WEEKLY TRAINING HISTORY SUMMARY ===


Weeks  Min_Training_Rows  Max_Training_Rows  \
Country   Horizon_Days                                                
Malaysia  1               105              18618              23472   
          5               104              18518              23328   
Singapore 1               105               5916               7413   
          5               104               5904               7389   

                        Total_Predictions  
Country   Horizon_Days                     
Malaysia  1                          4864  
          5                          4820  
Singapore 1                          1500  
          5                          1488


Completed.
